In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

print("Libraries Loaded Successfully!")

Libraries Loaded Successfully!


In [35]:
path = "D:/ecommerce-business-analysis/Data/"

customers  = pd.read_csv(path + "olist_customers_dataset.csv")
geolocation = pd.read_csv(path + "olist_geolocation_dataset.csv")
order_items = pd.read_csv(path + "olist_order_items_dataset.csv")
payments   = pd.read_csv(path + "olist_order_payments_dataset.csv")
reviews    = pd.read_csv(path + "olist_order_reviews_dataset.csv")
orders     = pd.read_csv(path + "olist_orders_dataset.csv")
products   = pd.read_csv(path + "olist_products_dataset.csv")
sellers    = pd.read_csv(path + "olist_sellers_dataset.csv")
category   = pd.read_csv(path + "product_category_name_translation.csv")

print("All 9 files loaded!")

All 9 files loaded!


In [22]:
def explore_df(df, name):
    print(f"\n{'='*50}")
    print(f" Dataset: {name}")
    print(f"   Shape   : {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"   Columns : {list(df.columns)}")
    print(f"\n   Missing Values:")
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) == 0:
        print(" No missing values!")
    else:
        print(missing)

explore_df(customers,   "Customers")
explore_df(geolocation, "Geolocation")
explore_df(order_items, "Order Items")
explore_df(payments,    "Payments")
explore_df(reviews,     "Reviews")
explore_df(orders,      "Orders")
explore_df(products,    "Products")
explore_df(sellers,     "Sellers")
explore_df(category,    "Category Translation")


 Dataset: Customers
   Shape   : 99441 rows, 5 columns
   Columns : ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

   Missing Values:
 No missing values!

 Dataset: Geolocation
   Shape   : 1000163 rows, 5 columns
   Columns : ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

   Missing Values:
 No missing values!

 Dataset: Order Items
   Shape   : 112650 rows, 7 columns
   Columns : ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

   Missing Values:
 No missing values!

 Dataset: Payments
   Shape   : 103886 rows, 5 columns
   Columns : ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

   Missing Values:
 No missing values!

 Dataset: Reviews
   Shape   : 99224 rows, 7 columns
   Columns : ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_commen

In [24]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print("Date columns converted!")
print(orders.dtypes)

Date columns converted!
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [25]:
print("Missing before:\n", orders.isnull().sum())

orders_clean = orders[orders['order_status'] == 'delivered'].copy()

orders_clean.dropna(subset=['order_delivered_customer_date'], inplace=True)

print(f"\nClean orders shape: {orders_clean.shape}")
print(f"   Removed {len(orders) - len(orders_clean)} incomplete rows")

Missing before:
 order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Clean orders shape: (96470, 8)
   Removed 2971 incomplete rows


In [26]:
products_clean = products.merge(
    category,
    on='product_category_name',
    how='left'
)
products_clean = products_clean[[
    'product_id',
    'product_category_name_english',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]]

print("Products cleaned!")
print(products_clean.head(3))

Products cleaned!
                         product_id product_category_name_english  \
0  1e9e8ef04dbcff4541ed26657ea517e5                     perfumery   
1  3aa071139cb16b67ca9e5dea641aaa2f                           art   
2  96bd76ec8810374ed1b65e291975717f                sports_leisure   

   product_weight_g  product_length_cm  product_height_cm  product_width_cm  
0             225.0               16.0               10.0              14.0  
1            1000.0               30.0               18.0              20.0  
2             154.0               18.0                9.0              15.0  


In [29]:
reviews_clean = reviews[[
    'order_id',
    'review_score'
]].copy()

reviews_clean.drop_duplicates(subset='order_id', inplace=True)

print("Reviews cleaned!")
print(reviews_clean['review_score'].value_counts())

Reviews cleaned!
review_score
5    57019
4    19044
1    11353
3     8124
2     3133
Name: count, dtype: int64


In [33]:
master = orders_clean.merge(customers,   on='customer_id',  how='left')
master = master.merge(order_items,       on='order_id',     how='left')
master = master.merge(payments,          on='order_id',     how='left')
master = master.merge(products_clean,    on='product_id',   how='left')
master = master.merge(sellers,           on='seller_id',    how='left')
master = master.merge(reviews_clean,     on='order_id',     how='left')

print(f"Master dataset ready!")
print(f"   Shape: {master.shape}")
print(f"\n   Columns:\n{list(master.columns)}")

Master dataset ready!
   Shape: (115030, 31)

   Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value', 'product_category_name_english', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'review_score']


In [36]:
master.to_csv("Data/master_dataset.csv", index=False)